# Secure Communication

This exercise deals with secure communication using TLS-based examples.

## Preparation

For shell command calls, the Python module ``subprocess`` is used (https://docs.python.org/3/library/subprocess.html).

Run the following code block to provide the functions ``execute_vm_command()`` and ``execute_stm32_command()`` within this notebook.

In [ ]:
import subprocess

def execute_vm_command(cmd):
    run = subprocess.run(cmd, universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)
    
def execute_stm32_command(cmd):
    ssh = 'sshpass -p "root" ssh root@192.168.13.13 '
    run = subprocess.run((ssh + cmd), universal_newlines=True, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(run.stdout)

Of course, all commands given in this tutorial can also be executed directly on within a Linux console.

If you want to open a terminal window, execute the next code block.

In [ ]:
execute_vm_command('gnome-terminal')

---

## Exercise 7.1: Basic TLS Server-Client Example

A big obstacle for secure device communication based on TLS is the necessity of X.509 certificates.

Within the following example, a secure machine-to-machine communication with mutual authentication based on TLS v1.3 should be established. 

Therefore, X.509 certificates for server and client are created that can be used for authentication in the TLS handshake.

### Create Server and Client Certificates

With the following commands, create a private key and a self-signed X.509 certificate for the server and the client. These can be used for TLS authentication later.

However, someone must have forgotten to set an important parameter!

Which parameter is it?

What would be a reasonable value for that parameter?

**Important Missing Parameter:**

*TODO*

**Reasonable Value:**

*TODO*

In [ ]:
execute_vm_command('openssl req -new -newkey rsa:2048 -days -nodes -x509 -keyout server.key -out server.crt -config configs/server.conf')

In [ ]:
execute_vm_command('openssl req -new -newkey rsa:2048 -days -nodes -x509 -keyout client.key -out client.crt -config configs/client.conf')

### Server Certificate Information

The following command yields detailed information about the created server certificate:

In [ ]:
execute_vm_command('openssl x509 -in server.crt -text -noout')

### Client Certificate Information

The following command yields detailed information about the created client certificate:

In [ ]:
execute_vm_command('openssl x509 -in client.crt -text -noout')

What do ``CN`` and ``O`` stand for and which values do they have in the two certificates?

**Meaning of ``CN``:**

*TODO*

**Meaning of ``O``:**

*TODO*

**Values of ``CN`` and ``O`` for Server and Client:**

Server:
* ``CN``: *TODO*
* ``O``: *TODO*

Client:
* ``CN``: *TODO*
* ``O``: *TODO*

### Copy Client Data

Create the directory ``/home/root/secure_communication`` on your STM32MP1 board for this exercise.

In [ ]:
execute_stm32_command('mkdir -p /home/root/secure_communication')

Use the following command to copy the Python application of a minimal TLS client, its private key, and the corresponding certificate to the STM32MP1 board. Also, the server certificate has to be placed on the board.

In [ ]:
execute_vm_command('sshpass -p "root" scp tls_minimal/tls_client.py client.key client.crt server.crt root@192.168.13.13:/home/root/secure_communication')

### Run Server Application

Open a terminal and execute ``./tls_minimal/tls_server.py`` to start a simple Python-based TLS server.
If everything started correctly, you should see the prompt ``Waiting for client ...``.

In [ ]:
execute_vm_command('gnome-terminal')

### Try to Run Client Application on STM32MP1 Board

Now, the client application on the STM32MP1 board can be started.

Open a terminal and connect to your board by SSH.

In [ ]:
execute_vm_command('gnome-terminal')

Then, navigate to the script's directory and execute it:

```bash
cd /home/root/secure_communication/
./tls_client.py

```

That does not seem to work...

What is the problem?

**Current TLS Connection Problem:**

*TODO*

### Time and Date

Get the current time and date on the STM32MP1 board:

In [ ]:
execute_stm32_command('date')
execute_stm32_command('/sbin/hwclock')

This data does not seem to be correct.

Use the tool ``date`` on your STM32MP1 board to set the correct time and date for today.

Example for Christmas 2000: ``date -s "Dec 24 2000 12:34:56"``

In [ ]:
execute_vm_command('gnome-terminal')

Check time and date again.

In [ ]:
execute_stm32_command('date')
execute_stm32_command('/sbin/hwclock')

Did it work?

**Notes on Setting Time and Date:**

*TODO*

### Start Server Again

Go back to the previously opened terminal and run the script ``tls_server.py`` again.

### Rerun TLS Client on STM32MP1 Board

Navigate to the script's directory on your board and execute it again:

```bash
cd /home/root/secure_communication/
./tls_client.py

```

Check, if the communication example ran as expected.

Which TLS version was negotiated and which cipher suite?

**Negotiated TLS Version:**

*TODO*

**Negotiated Cipher Suite:**

*TODO*

---

## Exercise 7.2: Using a Certificate Authority (CA)

The prior example used two completely unrelated certificates for the TLS communication. In this follow-up exercise, a self-signed CA certificate is used as the root for the server's and client's certificate chain.

### Create CA Key

First of all, a private RSA key is created for the CA.

In [ ]:
execute_vm_command('openssl genrsa -out ca.key 2048')

### Generate CA Certificate

The self-signed CA root certificate is created by signing it with the same key.

In [ ]:
execute_vm_command('openssl req -new -x509 -key ca.key -out ca.crt -days 1826 -config configs/ca.conf')

### Create Private Key and CSR for Client (STM32MP1 Board)

In the next steps, a key and a corresponding Certificate Signing Request (CSR) are generated for the client.

In [ ]:
execute_vm_command('openssl genrsa -out stm32mp1.key 2048')
execute_vm_command('openssl req -new -key stm32mp1.key -out stm32mp1.csr -config configs/stm32mp1.conf -extensions "v3_req"')

### Create Private Key and CSR for Server (VM)

The same has to be done for the server.

In [ ]:
execute_vm_command('openssl genrsa -out vmserver.key 2048')
execute_vm_command('openssl req -new -key vmserver.key -out vmserver.csr -config configs/vmserver.conf -extensions "v3_req"')

### Signing the CSRs by the CA

Finally, the CSRs can be signed by the CA, acting as a Trusted Third Party (TTP) in this scenario.

In [ ]:
execute_vm_command('openssl x509 -req -in stm32mp1.csr -CA ca.crt -CAkey ca.key -CAcreateserial -extfile configs/stm32mp1.conf -extensions v3_req -out stm32mp1.crt -days 730')
execute_vm_command('openssl x509 -req -in vmserver.csr -CA ca.crt -CAkey ca.key -CAcreateserial -extfile configs/vmserver.conf  -extensions v3_req -out vmserver.crt -days 730')

### Print Details of Created Certificates

The following commands print all information contained in the newly created certificates.

In [ ]:
execute_vm_command('openssl x509 -in stm32mp1.crt -text -noout')

In [ ]:
execute_vm_command('openssl x509 -in vmserver.crt -text -noout')

Discuss the printed information of the two certificates!

Name at least two differences to the certificates generated in the previous exercise!

**Differences to Prior Certificates:**

*TODO*

### Copy Client Data to STM32MP1 Board

On the STM32MP1 board, the client application, the CA certificate as well as the client key and certificate are required. The following command copies them.

In [ ]:
execute_vm_command('sshpass -p "root" scp tls_with_ca/tls_ca_client.py ca.crt stm32mp1.key stm32mp1.crt root@192.168.13.13:/home/root/secure_communication')

### Start Server Application

Like in the previous example, a terminal has to be opened, in which the server application ``tls_with_ca/tls_ca_server.py`` must be started.

Again, the prompt ``Waiting for client ...`` should appear.

In [ ]:
execute_vm_command('gnome-terminal')

### Run Client Application

Navigate to the client's directory on your board and execute it again:

```bash
cd /home/root/secure_communication/
./tls_ca_client.py

```

Well done!

... but wait ...

There is something clearly wrong! Someone must have made a critical implementation error in the server application.

Find out what the problem is, which lines of code in the server Python script are responsible for that, and how to fix it!

**Critical Problem:**

*TODO*

**Responsible Lines of Code in ``tls_with_ca/tls_ca_server.py``:**

*TODO*

**Fix for the Problem:**

*TODO*